# DATA 200 Midterm - Stack Overflow Developer Survey 2025

**Research Question:**  
*Does working with modern AI-related technologies in 2025 associate with higher compensation and different career trajectories, and does this relationship vary by experience level and geographic region?*

**Dataset:** Stack Overflow Developer Survey 2025 (`survey_results_public.csv`)

Key variables used:
- `ConvertedCompYearly`, `YearsCode`, `WorkExp` (continuous)
- `Country`, `DevType`, `EdLevel`, `AISelect` (categorical)
- `LanguageHaveWorkedWith`, `AIModelsHaveWorkedWith` (text, parsed with regex)


## 1. Data Loading and Cleaning

Loading the survey, keeping only the columns we actually need, and doing some basic cleaning. Compensation gets capped at the 99th percentile to knock out the extreme outliers. `YearsCode` and `WorkExp` have text entries like "Less than 1 year" that need to be converted to numbers.

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression, LogisticRegression
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13,
                     "axes.labelsize": 11, "xtick.labelsize": 9,
                     "ytick.labelsize": 9})

RAW_PATH = "survey_results_public.csv"
COLS = [
    "ResponseId", "Country", "Age", "EdLevel", "Employment", "DevType",
    "YearsCode", "WorkExp", "ConvertedCompYearly", "CompTotal",
    "LanguageHaveWorkedWith", "AIModelsHaveWorkedWith", "AISelect",
    "RemoteWork", "OrgSize"
]
raw = pd.read_csv(RAW_PATH, usecols=COLS, low_memory=False)

print(f"Raw shape: {raw.shape}")
raw.head(6)

Raw shape: (49191, 15)


,ResponseId,Age,EdLevel,Employment,WorkExp,YearsCode,DevType,OrgSize,RemoteWork,Country,CompTotal,LanguageHaveWorkedWith,AIModelsHaveWorkedWith,AISelect,ConvertedCompYearly
0,1,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,8.0,14.0,"Developer, mobile",20 to 99 employees,Remote,Ukraine,52800.0,Bash/Shell (all shells);Dart;SQL,openAI GPT (chatbot models);openAI Image gener...,"Yes, I use AI tools monthly or infrequently",61256.0
1,2,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,2.0,10.0,"Developer, back-end",500 to 999 employees,"Hybrid (some in-person, leans heavy to flexibi...",Netherlands,90000.0,Java,openAI GPT (chatbot models),"Yes, I use AI tools weekly",104413.0
2,3,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-em...",10.0,12.0,"Developer, front-end",NaN,NaN,Ukraine,2214000.0,Dart;HTML/CSS;JavaScript;TypeScript,Gemini (Flash general purpose models);openAI G...,"Yes, I use AI tools daily",53061.0
3,4,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,4.0,5.0,"Developer, back-end","10,000 or more employees",Remote,Ukraine,31200.0,Java;Kotlin;SQL,NaN,"Yes, I use AI tools weekly",36197.0
4,5,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",21.0,22.0,Engineering manager,NaN,NaN,Ukraine,60000.0,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,openAI GPT (chatbot models),"Yes, I use AI tools weekly",60000.0
5,6,45-54 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)","Independent contractor, freelancer, or self-em...",15.0,20.0,"Developer, back-end",NaN,NaN,Ukraine,120000.0,Java;Scala,NaN,"Yes, I use AI tools daily",120000.0


In [3]:
def coerce_years(series):
    s = series.copy().astype(str).str.strip()
    s = s.replace({"Less than 1 year": "0", "More than 50 years": "51", "nan": np.nan})
    return pd.to_numeric(s, errors="coerce")

In [4]:
df = raw.copy()
df["YearsCode"] = coerce_years(df["YearsCode"])
df["WorkExp"] = coerce_years(df["WorkExp"])
df["Comp"] = pd.to_numeric(df["ConvertedCompYearly"], errors="coerce")

# drop rows with no compensation
df = df.dropna(subset=["Comp"])

# cap at 99th percentile
cap = df["Comp"].quantile(0.99)
df  = df[df["Comp"] <= cap].copy()

# clean up education labels
ed_map = {
    "Primary/elementary school": "< High School",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "High School",
    "Some college/university study without earning a degree": "Some College",
    "Associate degree (A.A., A.S., etc.)": "Associate",
    "Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)": "Bachelor's",
    "Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master's",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "PhD/Professional",
    "Something else": "Other"
}
df["EdLevel"] = df["EdLevel"].map(ed_map).fillna(df["EdLevel"])

# simplify employment status
df["EmploySimple"] = df["Employment"].apply(
    lambda x: "Employed" if isinstance(x, str) and "Employed" in x
    else ("Freelance/Contract" if isinstance(x, str) and ("Independent" in x or "freelance" in x.lower())
    else ("Student" if isinstance(x, str) and "student" in x.lower()
    else "Other"))
)

# group countries - keep top 10, everything else is "Other"
top_countries = df["Country"].value_counts().head(10).index
df["CountryGroup"] = df["Country"].apply(lambda x: x if x in top_countries else "Other")

print(f"Analysis dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Compensation range (USD): ${df['Comp'].min():,.0f} - ${df['Comp'].max():,.0f}")
print(f"Median compensation: ${df['Comp'].median():,.0f}")
df[["Comp", "YearsCode", "WorkExp"]].describe().round(1)


Analysis dataset: 23,708 rows x 18 columns
Compensation range (USD): $1 - $440,856
Median compensation: $74,878


,Comp,YearsCode,WorkExp
count,23708.0,23600.0,23231.0
mean,87329.8,17.6,13.7
std,69468.6,11.0,10.1
min,1.0,1.0,1.0
25%,37410.5,9.0,6.0
50%,74878.0,15.0,11.0
75%,120000.0,25.0,20.0
max,440856.0,100.0,100.0
